In [ ]:
# clean_trainingdata.ipynb
# Clean sweep/training outputs on the pod. First run previews targets; set CONFIRM_DELETE=True to delete.
from pathlib import Path
import fnmatch
import os
import shutil
import subprocess
import time

REPO = Path('/workspace/stable-query-latent')
HEADS_DIR = REPO / 'VICReg_review/heads'
LOG_DIR = Path('/workspace/stable_query_latent_logs')
ARTIFACT_DIR = Path('/workspace/stable_query_latent_artifacts')

# Safety switch. Leave False for dry-run preview; change to True for one-click cleanup.
CONFIRM_DELETE = False

# Refuse cleanup when sweep/training processes are still running.
REFUSE_IF_TRAINING_RUNNING = True

# Default targets: sweep outputs and copied sweep artifacts. This intentionally keeps corpus files such as
# game_review_data/embedding_h5.h5 and game_review_data/build_new_gamedata/text_h5.h5.
# The previous version only listed a few fixed output dirs, so renamed smoke/final/cloud sweeps were missed.
DISCOVER_HEAD_OUTPUTS = True
HEAD_OUTPUT_NAME_PATTERNS = (
    'cloud_full_sweep*',
    'final_full_sweep*',
    'data_view_sweep*',
    'sweep_*',
    '*_smoke',
)
HEAD_OUTPUT_MARKERS = (
    'sweep_manifest.json',
    'data_view_sweep_summary.json',
    'DATA_VIEW_SWEEP_REPORT.md',
)

STATIC_TARGETS = [
    LOG_DIR / 'pipeline.log',
    ARTIFACT_DIR,
]

def discover_head_output_targets():
    if not DISCOVER_HEAD_OUTPUTS or not HEADS_DIR.exists():
        return []
    targets = []
    for child in HEADS_DIR.iterdir():
        if not child.is_dir():
            continue
        name_match = any(fnmatch.fnmatch(child.name, pattern) for pattern in HEAD_OUTPUT_NAME_PATTERNS)
        marker_match = any((child / marker).exists() for marker in HEAD_OUTPUT_MARKERS)
        if name_match or marker_match:
            targets.append(child)
    return sorted(targets)

def unique_targets(paths):
    targets = []
    seen = set()
    for path in paths:
        key = str(Path(path).resolve())
        if key not in seen:
            seen.add(key)
            targets.append(Path(path))
    return targets

TARGETS = unique_targets([*discover_head_output_targets(), *STATIC_TARGETS])

print('repo:', REPO)
print('confirm delete:', CONFIRM_DELETE)
print('targets:')
for target in TARGETS:
    print(' -', target)


In [ ]:
# Preview what will be removed and check for running training/sweep commands.
from dataclasses import dataclass

TRAINING_MARKERS = (
    'VICReg_review/sweep_cloud.py',
    'VICReg_review\\sweep_cloud.py',
    'VICReg_review/run_data_view_sweep.py',
    'VICReg_review\\run_data_view_sweep.py',
    'VICReg_review/train_vicreg_review_h5.py',
    'VICReg_review\\train_vicreg_review_h5.py',
    'train_vicreg_review_h5_paired.py',
    'VICReg_review/probe_worker.py',
    'VICReg_review\\probe_worker.py',
)

@dataclass
class TargetInfo:
    path: Path
    exists: bool
    kind: str
    files: int
    dirs: int
    bytes: int

def human_size(num_bytes: int) -> str:
    value = float(num_bytes)
    for unit in ['B', 'KiB', 'MiB', 'GiB', 'TiB']:
        if value < 1024 or unit == 'TiB':
            return f'{value:.1f} {unit}'
        value /= 1024

def summarize_path(path: Path) -> TargetInfo:
    path = Path(path)
    if not path.exists():
        return TargetInfo(path, False, 'missing', 0, 0, 0)
    if path.is_file() or path.is_symlink():
        try:
            size = path.stat().st_size
        except OSError:
            size = 0
        return TargetInfo(path, True, 'file', 1, 0, size)
    files = dirs = size = 0
    for root, dirnames, filenames in os.walk(path):
        dirs += len(dirnames)
        for filename in filenames:
            files += 1
            try:
                size += (Path(root) / filename).stat().st_size
            except OSError:
                pass
    return TargetInfo(path, True, 'dir', files, dirs, size)

def command_text_from_proc_dir(proc_dir: Path) -> str:
    try:
        raw = (proc_dir / 'cmdline').read_bytes()
        return raw.replace(b'\x00', b' ').decode('utf-8', errors='ignore')
    except Exception:
        return ''

def running_training_processes():
    found = []
    proc_root = Path('/proc')
    if not proc_root.exists():
        return found
    for proc_dir in proc_root.iterdir():
        if not proc_dir.name.isdigit():
            continue
        cmd = command_text_from_proc_dir(proc_dir)
        if cmd and any(marker in cmd for marker in TRAINING_MARKERS):
            found.append((int(proc_dir.name), cmd[:300]))
    return found

infos = [summarize_path(target) for target in TARGETS]
existing = [info for info in infos if info.exists]
total_bytes = sum(info.bytes for info in existing)

print('Cleanup preview')
print('existing targets:', len(existing), '/', len(infos))
print('total removable size:', human_size(total_bytes))
print()
for info in infos:
    status = 'FOUND' if info.exists else 'missing'
    print(f'{status:7} {info.kind:7} {human_size(info.bytes):>11} files={info.files:<7} dirs={info.dirs:<5} {info.path}')

running = running_training_processes()
print()
if running:
    print('Training/sweep processes detected:')
    for pid, cmd in running:
        print(f' - pid={pid} {cmd}')
else:
    print('No sweep/training process detected.')


In [ ]:
# Delete targets. Set CONFIRM_DELETE=True in Cell 1, then run all cells.
def is_inside(path: Path, root: Path) -> bool:
    try:
        path.resolve().relative_to(root.resolve())
        return True
    except ValueError:
        return False

def allowed_to_delete(path: Path) -> bool:
    path = Path(path)
    allowed_roots = [
        REPO / 'VICReg_review/heads',
        LOG_DIR,
        ARTIFACT_DIR,
    ]
    return any(path.resolve() == root.resolve() or is_inside(path, root) for root in allowed_roots)

def remove_target(path: Path):
    path = Path(path)
    if not path.exists():
        return 'missing'
    if not allowed_to_delete(path):
        raise RuntimeError(f'refusing to delete outside allowed roots: {path}')
    if path.is_dir() and not path.is_symlink():
        shutil.rmtree(path)
        return 'removed dir'
    path.unlink()
    return 'removed file'

if not CONFIRM_DELETE:
    print('Dry run only. Set CONFIRM_DELETE=True in Cell 1 to delete these targets.')
elif REFUSE_IF_TRAINING_RUNNING and running_training_processes():
    raise RuntimeError('Refusing cleanup because sweep/training processes are still running. Stop them first or set REFUSE_IF_TRAINING_RUNNING=False.')
else:
    start = time.time()
    deleted = []
    for target in TARGETS:
        status = remove_target(target)
        deleted.append((str(target), status))
        print(f'{status:12} {target}')
    print(f'cleanup finished in {time.time() - start:.1f}s')

    remaining = [summarize_path(target) for target in TARGETS]
    remaining_existing = [info for info in remaining if info.exists]
    print('remaining existing targets:', len(remaining_existing))
    for info in remaining_existing:
        print(f' - {human_size(info.bytes):>10} {info.path}')


In [ ]:
# Sanity check: expensive corpus/cache files intentionally kept.
KEEP = [
    REPO / 'game_review_data/embedding_h5.h5',
    REPO / 'game_review_data/embedding_h5.h5.incloud_manifest.json',
    REPO / 'game_review_data/build_new_gamedata/text_h5.h5',
    REPO / 'game_review_data/build_new_gamedata/text_h5.h5.manifest.json',
]

print('Kept files:')
for path in KEEP:
    if path.exists():
        info = summarize_path(path)
        print(f'FOUND   {human_size(info.bytes):>10} {path}')
    else:
        print(f'missing            {path}')
